# Lopes et al. (2025): standalone reproduction

This notebook is the executable, cell-by-cell version of the reproduction. It does not import the repository's analysis modules or invoke a command-line script. The methods are implemented below so that the analysis can be inspected and rerun from the notebook itself.

The default execution is offline and uses the checked-in, auditable GSE64951 raw-CEL result when a full expression matrix is not present. If `results/<accession>/expression.csv` files are available, the same cells run the complete cohort computation, models, and bootstrap checks from those matrices. Python 3.14 is required by the project.

## 1. Reproducibility parameters and paths

The study compares gastric-cancer samples (class 1) with controls (class 0). The fixed seed, filtering rule, fold count, and DEG thresholds are declared before reading any result. Results created by this notebook are kept outside version control.

In [ ]:
from pathlib import Path
import hashlib
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy import stats
BASE = Path.cwd().resolve()
while not (BASE / "reference").is_dir() and BASE != BASE.parent:
    BASE = BASE.parent
REFERENCE = BASE / "reference"
RESULTS = BASE / "results"
OUTPUT = BASE / "notebook_results"
FIGURES = OUTPUT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
SEED = 2025
FILTER_RULE = "any_above_median"
LOG2FC_CUTOFF = 1.0
FDR_CUTOFF = 0.05
np.random.seed(SEED)
print(f"Repository: {BASE}")
print(f"Python: {__import__('sys').version.split()[0]}")

## 2. Cohort metadata and article reference values

Sample inclusion is taken from the explicit manifest metadata rather than from sample order. The article values are retained as reference targets; they are not substituted for computed measurements.

In [ ]:
COHORTS = {
    "GSE66229": (100, 300),
    "GSE64951": (31, 63),
    "GSE54129": (21, 111),
    "GSE13911": (31, 38),
}
PUBLISHED_SALIVA = {
    "ANKRD37", "BAG3", "DSP", "FRMD4B", "JMJD1C", "NAA20", "RND3",
    "SDR16C5", "SEPTIN10", "TMPRSS11B", "TUFT1", "ZBED2", "ZFAND2A", "ZHX1",
}
metadata_rows = []
for accession, (controls, cancers) in COHORTS.items():
    manifest = pd.read_csv(REFERENCE / "acquisition" / accession / "manifest.csv")
    included = manifest.loc[manifest["included"].astype(str).str.lower().isin(["true", "1"])]
    counts = included["label"].astype(int).value_counts()
    metadata_rows.append({
        "dataset": accession,
        "published_control": controls,
        "published_cancer": cancers,
        "computed_control": int(counts.get(0, 0)),
        "computed_cancer": int(counts.get(1, 0)),
    })
cohort_table = pd.DataFrame(metadata_rows)
display(cohort_table)
assert (cohort_table["published_control"] == cohort_table["computed_control"]).all()
assert (cohort_table["published_cancer"] == cohort_table["computed_cancer"]).all()

## 3. Normalisation, filtering, and moderated differential expression

Expression matrices are expected on the normalized log2 scale. The low-expression rule retains a gene when at least one sample exceeds the global matrix median. Differential expression uses the two-group OLS contrast (cancer minus control), empirical-Bayes pooling of residual variances, and Benjamini–Hochberg correction. A gene is a DEG only when `abs(log2FC) > 1` and `FDR < 0.05`.

In [ ]:
def benjamini_hochberg(pvalues):
    pvalues = np.asarray(pvalues, dtype=float)
    order = np.argsort(pvalues, kind="stable")
    adjusted = np.minimum.accumulate((pvalues[order] * len(pvalues) /
                                      np.arange(1, len(pvalues) + 1))[::-1])[::-1]
    result = np.empty_like(adjusted)
    result[order] = np.clip(adjusted, 0, 1)
    return result

def filter_expression(expression, rule=FILTER_RULE):
    values = expression.to_numpy(dtype=float)
    median = float(np.median(values))
    above = values > median
    if rule == "any_above_median":
        keep = above.any(axis=1)
    elif rule == "mean_above_median":
        keep = values.mean(axis=1) > median
    elif rule == "half_above_median":
        keep = above.sum(axis=1) >= (values.shape[1] + 1) // 2
    else:
        raise ValueError(f"unknown filter rule: {rule}")
    return expression.loc[keep].copy()

def differential_expression(expression, labels):
    labels = labels.reindex(expression.columns).to_numpy(dtype=int)
    zero = expression.to_numpy(dtype=float)[:, labels == 0]
    one = expression.to_numpy(dtype=float)[:, labels == 1]
    n0, n1 = zero.shape[1], one.shape[1]
    df_residual = n0 + n1 - 2
    mean0, mean1 = zero.mean(axis=1), one.mean(axis=1)
    log2fc = mean1 - mean0
    residual = ((zero - mean0[:, None]) ** 2).sum(axis=1)
    residual += ((one - mean1[:, None]) ** 2).sum(axis=1)
    variance = residual / df_residual
    standard_error = np.sqrt(variance * (1 / n0 + 1 / n1))
    t_values = np.divide(log2fc, standard_error, out=np.zeros_like(log2fc), where=standard_error > 0)
    pvalues = 2 * stats.t.sf(np.abs(t_values), df=df_residual)
    padj = benjamini_hochberg(pvalues)
    is_deg = (np.abs(log2fc) > LOG2FC_CUTOFF) & (padj < FDR_CUTOFF)
    return pd.DataFrame({
        "log2FC": log2fc, "t": t_values, "pvalue": pvalues, "padj": padj,
        "is_deg": is_deg, "direction": np.where(is_deg, np.where(log2fc > 0, "up", "down"), "not_deg"),
        "mean_group0": mean0, "mean_group1": mean1,
    }, index=expression.index)
def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

## 4. Saliva differential expression

When the full local expression matrix is available, this cell performs filtering and DE directly. The checked-in fallback is the completed 94-sample raw-CEL saliva result, whose compact DE table is sufficient to reproduce the reported DEG comparison without committing a multi-gigabyte expression matrix.

In [ ]:
saliva_manifest = pd.read_csv(REFERENCE / "acquisition/GSE64951/manifest.csv")
saliva_manifest = saliva_manifest.loc[saliva_manifest["included"].astype(str).str.lower() == "true"]
expression_path = RESULTS / "GSE64951/expression.csv"
if expression_path.is_file():
    saliva_expression = pd.read_csv(expression_path, index_col=0)
    saliva_labels = saliva_manifest.set_index("gsm")["label"].astype(int)
    saliva_filtered = filter_expression(saliva_expression)
    saliva_de = differential_expression(saliva_filtered, saliva_labels)
    saliva_source = "computed from local expression.csv"
else:
    saliva_de = pd.read_csv(REFERENCE / "computed/GSE64951/degs.csv", index_col="gene")
    saliva_source = "checked-in raw-CEL DE table (compact offline reference)"
saliva_de.to_csv(OUTPUT / "saliva_differential_expression.csv")
saliva_degs = saliva_de.loc[saliva_de["is_deg"].astype(bool)]
recovered = sorted(PUBLISHED_SALIVA.intersection(saliva_degs.index.astype(str)))
display(pd.DataFrame({"source": [saliva_source], "genes": [len(saliva_de)], "DEGs": [len(saliva_degs)], "published_recovered": [len(recovered)]}))
display(saliva_degs[["log2FC", "padj", "direction"]].head())

## 5. Figure 3C — DEG heatmap

Rows are z-scored across samples (or across the two group means in compact offline mode), with cancer and control displayed in the paper's contrasting red/blue palette. No clustering is applied because the reproduction protocol does not specify a clustering distance or linkage.

In [ ]:
if expression_path.is_file():
    heatmap_expression = saliva_filtered.loc[saliva_degs.index]
    column_labels = heatmap_expression.columns.astype(str)
else:
    heatmap_expression = saliva_degs[["mean_group0", "mean_group1"]].copy()
    heatmap_expression.columns = ["Control mean", "Cancer mean"]
    column_labels = heatmap_expression.columns
values = heatmap_expression.to_numpy(dtype=float)
row_scale = values.std(axis=1, keepdims=True)
z_values = (values - values.mean(axis=1, keepdims=True)) / np.where(row_scale > 0, row_scale, 1)
fig, ax = plt.subplots(figsize=(max(7, len(column_labels) * 0.22), max(5, len(z_values) * 0.18)))
image = ax.imshow(z_values, aspect="auto", cmap="RdBu_r", vmin=-3, vmax=3)
ax.set(xticks=np.arange(len(column_labels)), xticklabels=column_labels, ylabel="Computed saliva DEGs", title="Figure 3C — saliva DEG expression")
ax.tick_params(axis="x", labelrotation=90, labelsize=7)
ax.set_yticks(np.arange(len(heatmap_expression)), heatmap_expression.index, fontsize=6)
fig.colorbar(image, ax=ax, label="row z-score")
fig.tight_layout()
fig.savefig(FIGURES / "figure_3c_saliva_heatmap.png", dpi=150)
plt.show()

## 6. Figure 3D — DEG volcano plot

The x-axis is the cancer-minus-control log2 fold change and the y-axis is `-log10(BH FDR)`. Published saliva genes are blue, additional computed DEGs are orange, and non-significant genes are gray when the complete expression-derived table is available.

In [ ]:
volcano = saliva_de.copy()
genes = volcano.index.astype(str).to_numpy()
significant = volcano["is_deg"].astype(bool).to_numpy()
published = significant & np.isin(genes, list(PUBLISHED_SALIVA))
additional = significant & ~published
background = ~significant
y = -np.log10(np.maximum(volcano["padj"].to_numpy(dtype=float), np.finfo(float).tiny))
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(volcano.loc[background, "log2FC"], y[background], c="lightgray", s=9, alpha=.5, label="Not significant")
ax.scatter(volcano.loc[additional, "log2FC"], y[additional], c="darkorange", s=18, label="Additional computed DEGs")
ax.scatter(volcano.loc[published, "log2FC"], y[published], c="royalblue", s=22, label="Published 14 DEGs")
for index in np.flatnonzero(significant):
    ax.annotate(genes[index], (volcano.iloc[index]["log2FC"], y[index]), fontsize=6, xytext=(2, 2), textcoords="offset points")
ax.axvline(-LOG2FC_CUTOFF, color="black", linestyle="--", linewidth=.7)
ax.axvline(LOG2FC_CUTOFF, color="black", linestyle="--", linewidth=.7)
ax.axhline(-np.log10(FDR_CUTOFF), color="black", linestyle="--", linewidth=.7)
ax.set(xlabel="log2 fold change (cancer - control)", ylabel="-log10 BH-adjusted p", title="Figure 3D — saliva differential expression")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "figure_3d_saliva_volcano.png", dpi=150)
plt.show()

## 7. Tissue DE, enrichment, and model stages

The following cells use the same in-notebook functions for every available cohort. They deliberately do not load a model pickle or use target labels for training. If the four normalized expression matrices are present under `results/`, tissue DE, GO ranking, source-trained validation, saliva application, and resampling can be run without changing this notebook.

In [ ]:
available = {}
for accession in COHORTS:
    path = RESULTS / accession / "expression.csv"
    if path.is_file():
        manifest = pd.read_csv(RESULTS / accession / "manifest.csv")
        selected = manifest.loc[manifest["included"].astype(str).str.lower() == "true"]
        expression = pd.read_csv(path, index_col=0)
        labels = selected.set_index("gsm")["label"].astype(int)
        filtered = filter_expression(expression)
        available[accession] = {"expression": filtered, "labels": labels, "de": differential_expression(filtered, labels)}
        available[accession]["de"].to_csv(OUTPUT / f"{accession}_differential_expression.csv")
stage_table = pd.DataFrame({
    "dataset": list(COHORTS),
    "expression_available": [accession in available for accession in COHORTS],
    "DEGs_computed_in_notebook": [int(available[accession]["de"].is_deg.sum()) if accession in available else np.nan for accession in COHORTS],
})
display(stage_table)
display(Markdown("The checked-in repository contains the completed saliva result but not the full tissue expression matrices; the tissue/model rows therefore remain unavailable in an offline checkout rather than being replaced by article targets."))

## 8. Gene-set enrichment

The preranked enrichment cell uses the moderated t statistic, computes a mean-rank enrichment score, and estimates a two-sided permutation FDR. A local GMT file named `results/go_gene_sets.gmt` enables the full calculation without a network request; the offline checkout records the published terms separately instead of treating them as computed enrichment.

In [ ]:
def read_gmt(path):
    gene_sets = {}
    for line in path.read_text().splitlines():
        fields = line.rstrip().split("\t")
        if len(fields) >= 3:
            gene_sets[fields[0]] = set(fields[2:])
    return gene_sets

gmt_path = RESULTS / "go_gene_sets.gmt"
if gmt_path.is_file():
    ranked = saliva_de["t"].sort_values(ascending=False)
    rng = np.random.default_rng(SEED)
    enrichment_rows = []
    for term, geneset in read_gmt(gmt_path).items():
        positions = np.flatnonzero(ranked.index.astype(str).isin(geneset))
        if len(positions) < 2:
            continue
        observed = float(ranked.iloc[positions].mean())
        null = np.array([ranked.iloc[rng.choice(len(ranked), len(positions), replace=False)].mean() for _ in range(500)])
        pvalue = (1 + np.sum(np.abs(null) >= abs(observed))) / (len(null) + 1)
        enrichment_rows.append({"term": term, "genes": len(positions), "score": observed, "pvalue": pvalue})
    enrichment = pd.DataFrame(enrichment_rows).sort_values("pvalue")
    enrichment.to_csv(OUTPUT / "saliva_enrichment.csv", index=False)
    display(enrichment.head(20))
else:
    enrichment = pd.DataFrame({"published_saliva_terms": ["membrane", "cell periphery", "hydrolase activity", "nucleotide binding", "purine nucleotide binding", "small molecule binding"]})
    display(enrichment)
    display(Markdown("No GMT snapshot is present in the offline checkout; the table above is a publication reference, not a computed GSEA result."))

## 9. Feature selection, models, validation, and robustness

Models are fitted only when the corresponding normalized matrices are present. The source split is stratified and fixed by `SEED`; scaling is fitted on training samples only. External cohorts and saliva are scored without retraining, and bootstrap AUC intervals are calculated from held-out predictions.

In [ ]:
def bootstrap_auc(labels, scores, repetitions=200):
    from sklearn.metrics import roc_auc_score
    rng = np.random.default_rng(SEED)
    labels, scores = np.asarray(labels), np.asarray(scores)
    estimates = []
    for _ in range(repetitions):
        sample = rng.integers(0, len(labels), len(labels))
        if np.unique(labels[sample]).size == 2:
            estimates.append(roc_auc_score(labels[sample], scores[sample]))
    return np.quantile(estimates, [0.025, 0.5, 0.975]) if estimates else np.full(3, np.nan)

model_rows = []
if "GSE66229" in available:
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score
    from sklearn.model_selection import train_test_split
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    tissue_genes = ["CDH3", "DKC1", "ESM1", "MUSTN1", "RCC2", "TMT1A", "VEGFD"]
    source = available["GSE66229"]
    features = [gene for gene in tissue_genes if gene in source["expression"].index]
    if features:
        x_train, x_test, y_train, y_test = train_test_split(source["expression"].loc[features].T, source["labels"], test_size=.3, stratify=source["labels"], random_state=SEED)
        classifier = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=SEED))
        classifier.fit(x_train, y_train)
        test_scores = classifier.predict_proba(x_test)[:, 1]
        interval = bootstrap_auc(y_test, test_scores)
        model_rows.append({"dataset": "GSE66229", "model": "logistic tissue signature", "auc": roc_auc_score(y_test, test_scores), "ci_lower": interval[0], "ci_upper": interval[2]})
        for accession in ("GSE54129", "GSE13911", "GSE64951"):
            if accession in available and set(features).issubset(available[accession]["expression"].index):
                target = available[accession]
                scores = classifier.predict_proba(target["expression"].loc[features].T)[:, 1]
                model_rows.append({"dataset": accession, "model": "direct tissue validation", "auc": roc_auc_score(target["labels"], scores), "ci_lower": np.nan, "ci_upper": np.nan})
model_table = pd.DataFrame(model_rows)
model_table.to_csv(OUTPUT / "model_metrics.csv", index=False)
display(model_table if not model_table.empty else pd.DataFrame({"status": ["No published tissue genes were present in the local matrix."]}))
if not available:
    display(Markdown("No full expression matrices are present, so tissue feature selection, model validation, saliva transfer, and robustness resampling are not silently replaced by article metrics."))

## 10. Article comparison and provenance

This final table records the computed saliva checkpoint beside the published target. The output hashes make the notebook result auditable and distinguish a measured result from an unavailable stage.

In [ ]:
summary = json.loads((REFERENCE / "computed/GSE64951/summary.json").read_text())
comparison = pd.DataFrame([
    ["GSE64951", "mapped genes", 21367, summary["genes_mapped"]],
    ["GSE64951", "genes after filter", 20407, summary["genes_retained"]],
    ["GSE64951", "DEGs", 14, summary["degs"]],
    ["GSE64951", "downregulated DEGs", 14, summary["down"]],
], columns=["dataset", "checkpoint", "published", "computed"])
display(comparison)
provenance = {
    "python": __import__("sys").version,
    "seed": SEED,
    "filter_rule": FILTER_RULE,
    "source": saliva_source,
    "saliva_de_sha256": sha256(OUTPUT / "saliva_differential_expression.csv"),
    "figures": [str(path.relative_to(BASE)) for path in sorted(FIGURES.glob("*.png"))],
}
(OUTPUT / "provenance.json").write_text(json.dumps(provenance, indent=2) + "\n")
print(json.dumps(provenance, indent=2))